# SAE Feature Decomposition

Use pre-trained Sparse Autoencoders to decompose a model's activations into interpretable features.

No SAELens dependency needed — interpkit loads the weights directly from HuggingFace.

In [1]:
import interpkit

model = interpkit.load("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

## Load a pre-trained SAE from HuggingFace

Pass a HuggingFace repo ID — interpkit downloads the weights directly via `safetensors`, no SAELens dependency required.

The shorthand `"<org>/<repo>/<subfolder>"` is supported for repos that store one SAE per layer in subdirectories (which is the common SAELens layout). Below we load the layer-8 residual stream SAE from `jbloom/GPT2-Small-SAEs-Reformatted` (≈150 MB; cached in `~/.cache/huggingface/hub` after the first download).

In [2]:
from interpkit.ops.sae import load_sae

sae = load_sae("jbloom/GPT2-Small-SAEs-Reformatted/blocks.8.hook_resid_pre")
print(f"SAE: {sae.d_in} -> {sae.d_sae} features")

SAE: 768 -> 24576 features


## Decompose activations

The SAE we just downloaded was trained on `blocks.8.hook_resid_pre` — the residual stream **entering** block 8. In HF GPT-2 that's the output of `transformer.h.7`, so that's where we hook.

In [3]:
result = model.features(
    "The capital of France is",
    at="transformer.h.7",
    sae=sae,
    top_k=15,
)

SAE Features — transformer.h.7 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Active: 24057 / 24576  │  Sparsity: 76.57%  │  Recon error: 66306.9844

 Rank   Feature   Activation                               
 ────────────────────────────────────────────────────────── 
     1     21490     141.1478   ████████████████████████    
     2     11746     105.9051   ██████████████████          
     3     14957     102.5148   █████████████████           
     4      6991     101.6643   █████████████████           
     5     11533      97.5155   ████████████████▌           
     6     20644      97.4069   ████████████████▌           
     7      4078      95.3759   ████████████████            
     8      6963      94.4317   ████████████████            
     9      8470      93.2983   ███████████████▌            
    10     10407      92.6632   ███████████████▌            
    11       630      92.5688   ███████████████▌            
    12     15526      91.7017   ███████████████▌            
    13      1013      91.3822   ███████████████▌            
    14     13997      91.1117   ███████████████             
    15      6784      90.2340   ███████████████

╭──────────── Top feature ─────────────╮
│  Feature 21490  activation 141.1478  │
╰──────────────────────────────────────╯

## Inspect the result dict

In [4]:
print(f"Reconstruction error: {result['reconstruction_error']:.4f}")
print(f"Sparsity: {result['sparsity']:.2%}")
print(f"Active features: {result['num_active_features']} / {result['total_features']}")
print("\nTop 5 features:")
for idx, val in result['top_features'][:5]:
    print(f"  Feature {idx}: activation = {val:.4f}")

Reconstruction error: 66306.9844
Sparsity: 76.57%
Active features: 24057 / 24576

Top 5 features:
  Feature 21490: activation = 141.1478
  Feature 11746: activation = 105.9051
  Feature 14957: activation = 102.5148
  Feature 6991: activation = 101.6643
  Feature 11533: activation = 97.5155


## Pass the SAE directly to `model.features`

You don't have to call `load_sae` yourself — `model.features(..., sae=...)` will load it for you. Either form works:

```python
# Shorthand
model.features(
    "The capital of France is",
    at="transformer.h.7",
    sae="jbloom/GPT2-Small-SAEs-Reformatted/blocks.8.hook_resid_pre",
)

# Or with explicit subfolder
model.features(
    "The capital of France is",
    at="transformer.h.7",
    sae="jbloom/GPT2-Small-SAEs-Reformatted",
    sae_subfolder="blocks.8.hook_resid_pre",
)
```

## Compare features across prompts

See which features activate differently for related but distinct inputs.

In [5]:
prompts = [
    "The capital of France is",
    "The capital of Germany is",
    "The largest city in Japan is",
]

for prompt in prompts:
    print(f"\n{'='*60}")
    print(f"Prompt: {prompt}")
    r = model.features(prompt, at="transformer.h.7", sae=sae, top_k=5)


Prompt: The capital of France is


SAE Features — transformer.h.7 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Active: 24057 / 24576  │  Sparsity: 76.57%  │  Recon error: 66306.9844

 Rank   Feature   Activation                               
 ────────────────────────────────────────────────────────── 
     1     21490     141.1478   ████████████████████████    
     2     11746     105.9051   ██████████████████          
     3     14957     102.5148   █████████████████           
     4      6991     101.6643   █████████████████           
     5     11533      97.5155   ████████████████▌

╭──────────── Top feature ─────────────╮
│  Feature 21490  activation 141.1478  │
╰──────────────────────────────────────╯

...


Prompt: The largest city in Japan is


SAE Features — transformer.h.7 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Active: 24090 / 24576  │  Sparsity: 78.92%  │  Recon error: 55391.1523

 Rank   Feature   Activation                               
 ────────────────────────────────────────────────────────── 
     1     21490     117.6231   ████████████████████████    
     2     11746      88.6264   ██████████████████          
     3      6991      85.8179   █████████████████▌          
     4     14957      85.4290   █████████████████           
     5     11533      81.5220   ████████████████▌

╭──────────── Top feature ─────────────╮
│  Feature 21490  activation 117.6231  │
╰──────────────────────────────────────╯

## Offline fallback — synthetic SAE

If you can't download from HuggingFace (no network, air-gapped environment, etc.) you can construct a synthetic SAE from raw tensors. The features won't be meaningful — random weights produce random "concepts" — but the API surface is identical, so this is useful for tests, CI, or bootstrapping a custom SAE you trained yourself.

In [6]:
import torch

from interpkit.ops.sae import load_sae_from_tensors

d_in = 768
d_sae = 4096

synthetic = load_sae_from_tensors(
    W_enc=torch.randn(d_in, d_sae) * 0.01,
    W_dec=torch.randn(d_sae, d_in) * 0.01,
    b_enc=torch.zeros(d_sae),
    b_dec=torch.zeros(d_in),
)
print(f"Synthetic SAE: {synthetic.d_in} -> {synthetic.d_sae} features")

result = model.features(
    "The capital of France is",
    at="transformer.h.7",
    sae=synthetic,
    top_k=5,
)

Synthetic SAE: 768 -> 4096 features


SAE Features — transformer.h.7 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Active: 3177 / 4096  │  Sparsity: 50.91%  │  Recon error: 705.3109

 Rank   Feature   Activation                               
 ────────────────────────────────────────────────────────── 
     1       426      25.5219   ████████████████████████    
     2      1024      24.0815   ██████████████████████▌     
     3      1553      21.0590   ███████████████████▌        
     4        29      20.8444   ███████████████████▌        
     5      2611      20.4354   ███████████████████

╭─────────── Top feature ───────────╮
│  Feature 426  activation 25.5219  │
╰───────────────────────────────────╯